# XGBoost Inflation Forecasting — Harmonised Specification

This notebook implements the XGBoost estimation pipeline for the thesis  
*"Same Same, But Different? Comparing SHAP and TVP Decompositions of Norwegian Inflation Forecasts"*.

The model forecasts Norwegian year-on-year CPI inflation **3 months ahead (h = 3)** using an  
expanding-window real-time design. Two specifications are estimated in parallel:

> ⚠️ **Requirement:** `master_data.csv` must be located in the same directory as this notebook before running any cells.

| Spec | Oil / trade denomination | Output prefix |
|------|--------------------------|---------------|
| `main` | NOK | `xgb_v8_harmonized_` |
| `usd_rob` | USD (robustness check) | `xgb_harmonized_robustness_` |

**Pipeline overview:**
1. Feature engineering — 9 macroeconomic variables, lag-1 information set
2. Hyperparameter tuning — Optuna TPE, 100 trials, re-tuned every 3 origins
3. Expanding-window backtest — SHAP attribution computed at each forecast origin
4. Block aggregation — SHAP values summed into 7 economic blocks
5. Output export — CSV + JSON files read by the R comparison script

## 1. Imports and Setup

Standard scientific stack plus three domain-specific libraries:
- **`xgboost`** — gradient-boosted tree ensemble (Chen & Guestrin, 2016)
- **`shap`** — exact TreeExplainer for additive feature attribution (Lundberg & Lee, 2017)
- **`optuna`** — Bayesian hyperparameter optimisation with TPE sampler (Akiba et al., 2019)

In [ ]:

from __future__ import annotations
import os, json, warnings
from pathlib import Path
from typing import Dict, List, Tuple
 
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import optuna
 
from optuna.samplers import TPESampler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Configuration

All global constants are defined here so that changing one value (e.g. `HORIZON` or `TEST_START`)  
propagates consistently through the entire pipeline.

**Key design choices:**
- `HORIZON = 3` — matches the Walker TVP model for a fair cross-model comparison
- `TEST_START = 2020-01-01` — start of the out-of-sample evaluation period
- `RETUNE_EVERY = 3` — Optuna re-runs every 3 forecast origins (~quarterly), balancing  
  adaptation to changing conditions against compute cost
- `MIN_TRAIN_N = 50` — minimum training observations before the first forecast is made;  
  harmonised with the Walker specification

`FEATURE_GROUPS` defines the mapping from individual variables to the 7 economic blocks  
used in the attribution comparison. `REGIME_CUTS` partitions the evaluation period into  
four descriptive macroeconomic episodes (COVID, Energy Crisis, Disinflation, Normalisation).

In [ ]:
DATA_FILE     = "master_data.csv"
RESULTS_DIR   = Path("results_harmonized")
FIGURES_DIR   = Path("figures_harmonized")
 
HORIZON       = 3                  # h-step-ahead forecast (months)
TEST_START    = "2020-01-01"
RETUNE_EVERY  = 3                  # months between Optuna re-tunes
OPTUNA_TRIALS = 100
MIN_TRAIN_N   = 50                 # unified with Walker (was 36 in original)
RANDOM_STATE  = 42
 
# Spec catalogue: name -> (oil_col, import_col, eksport_col, output_prefix)
SPECS = {
    "main":    ("oil_price_nok", "import",     "eksport",     "xgb_v8_harmonized_"),
    "usd_rob": ("oljepris_USD",  "import_USD", "eksport_USD", "xgb_harmonized_robustness_"),
}
 
FEATURE_GROUPS = {
    "AR (inflation)":  ["kpi_yoy_lag1"],
    "PPI / cost-push": ["ppi_yoy_lag1"],
    "Monetary policy": ["rente_lag1"],
    "FX":              ["usd_nok_lag1", "eur_nok_lag1"],
    "Oil":             ["oil_yoy_lag1"],
    "Trade":           ["import_yoy_lag1", "eksport_yoy_lag1"],
    "Labour market":   ["unemp_lag1"],
}
BLOCK_NAMES = list(FEATURE_GROUPS.keys())
 
REGIME_CUTS = [
    ("2021-06-01", "COVID (2020–2021)"),
    ("2023-01-01", "Energy Crisis"),
    ("2024-06-01", "Disinflation"),
    (None,         "Normalization"),
]
 
np.random.seed(RANDOM_STATE)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
 
 

## 3. Helper Functions

### `assign_regime` / `feature_group`
Utility lookups used to label each forecast origin and each feature throughout the pipeline.

### `make_recency_weights(n, half_life)`
Computes exponential decay weights over the training window:
$$w_t = \exp\!\left(-\frac{\ln 2}{\text{HL}} \cdot a_t\right)$$
where $a_t$ is the age of observation $t$ in months (0 = most recent).  
Weights are normalised so their mean equals 1, preserving the effective sample size  
interpretation of the loss function. **Important:** weights are recomputed fresh at each  
expanding-window origin and are never normalised across origins.

### `build_features(df, oil_col, import_col, eksport_col, h)`
Constructs the 9-variable information set shared by both XGBoost and Walker.  
All features are lagged by **at least 1 month** relative to the forecast origin,  
satisfying the h = 3 no-leakage requirement (predictors observed at $t-1$, target realised at $t+h$).  
YoY transformations are 12-month log-differences of the lagged level series.

The target `yoy_inflation_h3` is $\pi_{t+3}$, i.e. year-on-year CPI inflation three months ahead.

In [ ]:
def assign_regime(date: pd.Timestamp) -> str:
    for cutoff, label in REGIME_CUTS:
        if cutoff is None or date < pd.Timestamp(cutoff):
            return label
    return "Unassigned"
 
 
def feature_group(col: str) -> str:
    for g, members in FEATURE_GROUPS.items():
        if col in members:
            return g
    return "Other"
 
 
def make_recency_weights(n: int, half_life: float) -> np.ndarray:
    """Exponential decay weights normalised so the mean weight is 1."""
    decay = np.log(2) / half_life
    ages = np.arange(n - 1, -1, -1, dtype=float)
    w = np.exp(-decay * ages)
    return w / w.mean()
 
 
def build_features(
    df: pd.DataFrame,
    oil_col: str,
    import_col: str,
    eksport_col: str,
    h: int,
) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    """
    Lag-1 information set used by both Walker and XGBoost. At forecast origin t
    every feature is observed at t-1; YoY transformations therefore use t-13
    versus t-1. Target is YoY KPI at t+h (levels at t+h vs t+h-12).
    """
    out = pd.DataFrame(index=df.index)
 
    kpi_yoy_raw     = df["kpi"].pct_change(12) * 100
    ppi_yoy_raw     = df["ppi"].pct_change(12) * 100
    oil_yoy_raw     = df[oil_col].pct_change(12) * 100
    import_yoy_raw  = df[import_col].pct_change(12) * 100
    eksport_yoy_raw = df[eksport_col].pct_change(12) * 100
 
    out["kpi_yoy_lag1"]     = kpi_yoy_raw.shift(1)
    out["ppi_yoy_lag1"]     = ppi_yoy_raw.shift(1)
    out["oil_yoy_lag1"]     = oil_yoy_raw.shift(1)
    out["usd_nok_lag1"]     = df["usd_nok"].shift(1)
    out["eur_nok_lag1"]     = df["eur_nok"].shift(1)
    out["import_yoy_lag1"]  = import_yoy_raw.shift(1)
    out["eksport_yoy_lag1"] = eksport_yoy_raw.shift(1)
    out["unemp_lag1"]       = df["unemployment"].shift(1)
    out["rente_lag1"]       = df["rente"].shift(1)
 
    target = kpi_yoy_raw.shift(-h).rename(f"yoy_inflation_h{h}")
    return out, target, kpi_yoy_raw.rename("kpi_yoy_raw")
 
 

## 4. Hyperparameter Tuning (Optuna)

`tune_xgb` runs an Optuna study that optimises **XGBoost hyperparameters and  
the recency-weight half-life** using 3-fold time-series cross-validation within the  
current training window.

**Search space summary:**

| Parameter | Range | Scale |
|-----------|--------|-------|
| `half_life` | 6 – 60 months | log |
| `n_estimators` | 50 – 250 | linear |
| `learning_rate` | 0.01 – 0.25 | log |
| `max_depth` | 2 – 4 | linear |
| `min_child_weight` | 5 – 30 | linear |
| `reg_alpha` / `reg_lambda` | regularisation | log |

The inner CV uses `TimeSeriesSplit(n_splits=3)`, which preserves the ordering  
of the data and prevents any future information from leaking into the validation set.

The TPE sampler (`seed=42`) is used throughout for reproducibility.

In [ ]:
def tune_xgb(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    n_trials: int,
    random_state: int,
) -> Tuple[Dict, float, float]:
    """Optuna search; half-life and standard XGB knobs jointly tuned."""
    def objective(trial):
        hl = trial.suggest_float("half_life", 6, 60, log=True)
        w  = make_recency_weights(len(X_tr), hl)
        params = {
            "objective":         "reg:squarederror",
            "n_estimators":      trial.suggest_int("n_estimators", 50, 250),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.25, log=True),
            "max_depth":         trial.suggest_int("max_depth", 2, 4),
            "min_child_weight":  trial.suggest_int("min_child_weight", 5, 30),
            "subsample":         trial.suggest_float("subsample", 0.6, 0.95),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
            "gamma":             trial.suggest_float("gamma", 0.0, 2.0),
            "random_state":      random_state,
            "n_jobs":            -1,
        }
        rmses = []
        for tr_idx, val_idx in TimeSeriesSplit(n_splits=3).split(X_tr):
            m = xgb.XGBRegressor(**params)
            m.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx],
                  sample_weight=w[tr_idx], verbose=False)
            preds = m.predict(X_tr.iloc[val_idx])
            rmses.append(np.sqrt(mean_squared_error(y_tr.iloc[val_idx], preds)))
        return float(np.mean(rmses))
 
    study = optuna.create_study(direction="minimize",
                                sampler=TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = dict(study.best_params)
    hl   = best.pop("half_life")
    return best, hl, float(study.best_value)
 

### Recency weighting

The recency weights take the form $w_t = \exp(-\lambda \cdot a_t)$, where $a_t$ is the age of observation $t$ in months and $\lambda = \ln(2) / h$ is determined by the half-life $h$. The half-life is treated as a hyperparameter and tuned jointly with the standard XGBoost parameters using Optuna with log-uniform sampling over $[6, 60]$ months.

The lower bound is motivated by the fact that very short half-lives produce an effective sample concentrated on the most recent year of data, which is insufficient for stable tree-based estimation given the dimensionality of the feature space. The upper bound reflects the fact that with training samples of approximately $240$ observations, half-lives beyond five years yield nearly flat weights, so further extending the search range adds no expressive capacity. The log scale ensures that the optimiser allocates trials proportionally across orders of magnitude rather than oversampling the upper end of the interval.

## 5. Expanding-Window Backtest

`run_backtest` is the main estimation loop. For each forecast origin $t$ in the test period:

1. **Split** — training set is all observations strictly before $t$ (`Date < origin`, not `<=`)  
2. **Re-tune** (every 3 origins) — Optuna re-runs on the expanded training window  
3. **Fit** — XGBoost is trained with recency weights on the full training set  
4. **Predict** — one-step forecast at horizon h = 3  
5. **SHAP** — `TreeExplainer` computes exact additive attributions for the forecast  

**SHAP reconstruction check:** `shap_base + sum(shap_values) ≈ predicted_raw` is logged  
as `reconstruction_error` at each origin. This verifies that the attribution is  
internally consistent — a necessary condition before comparing with Walker contributions.

The random walk benchmark (`y_rw = kpi_yoy_lag1`) is stored alongside the XGBoost  
forecast at each origin so that DM tests can be computed in post-processing.

In [ ]:
def run_backtest(
    X_full: pd.DataFrame,
    y_full: pd.Series,
    feature_cols: List[str],
    test_start: str,
    retune_every: int,
    optuna_trials: int,
    min_train: int,
    random_state: int,
):
    """Sequential by design: each origin re-fits XGB on the expanding train set."""
    test_dates = X_full.index[X_full.index >= test_start]
    print(f"Test period: {test_dates[0].date()} – {test_dates[-1].date()} "
          f"({len(test_dates)} origins)")
 
    results, shap_records, tuning_log = [], [], []
    current_params, current_hl = None, 12.0
 
    for i, test_date in enumerate(test_dates):
        X_tr = X_full[X_full.index < test_date]
        y_tr = y_full[y_full.index < test_date]
        X_te = X_full.loc[[test_date]]
        y_te = float(y_full.loc[test_date])
 
        if len(X_tr) < min_train:
            continue
 
        # Re-tune every k months (or on the first eligible origin)
        if i % retune_every == 0 or current_params is None:
            print(f"[{test_date.date()}] Re-tuning ({optuna_trials} trials, "
                  f"train={len(X_tr)})...", end=" ", flush=True)
            current_params, current_hl, best_cv = tune_xgb(
                X_tr, y_tr, n_trials=optuna_trials, random_state=random_state
            )
            print(f"CV RMSE={best_cv:.4f}, HL={current_hl:.1f}m")
            tuning_log.append({
                "date": test_date, "cv_rmse": best_cv,
                "train_size": len(X_tr), "half_life": current_hl,
                **current_params,
            })
 
        w_tr  = make_recency_weights(len(X_tr), half_life=current_hl)
        model = xgb.XGBRegressor(random_state=random_state, n_jobs=-1,
                                 **current_params)
        model.fit(X_tr, y_tr, sample_weight=w_tr, verbose=False)
 
        y_hat = float(model.predict(X_te)[0])
        y_rw  = float(X_te["kpi_yoy_lag1"].iloc[0])
        results.append({
            "date": test_date, "actual": y_te,
            "predicted_raw": y_hat, "error_raw": y_te - y_hat,
            "y_rw": y_rw, "rw_error": y_te - y_rw,
            "train_size": len(X_tr), "half_life": current_hl,
            "regime": assign_regime(test_date),
        })
 
        # SHAP at the forecast origin (TreeExplainer is exact for tree models)
        explainer   = shap.TreeExplainer(model)
        shap_values = np.asarray(explainer.shap_values(X_te)).reshape(-1)
        shap_base   = float(np.asarray(explainer.expected_value).reshape(-1)[0])
        recon_pred  = shap_base + float(shap_values.sum())
        row = {
            "date": test_date, "shap_base": shap_base,
            "predicted_raw": y_hat, "pred_reconstructed": recon_pred,
            "reconstruction_error": recon_pred - y_hat,
            "actual": y_te, "regime": assign_regime(test_date),
        }
        for j, col in enumerate(feature_cols):
            row[col] = float(shap_values[j])
        shap_records.append(row)
 
        if (i + 1) % 12 == 0 or i == len(test_dates) - 1:
            recent = results[-min(12, len(results)):]
            recent_rmse = np.sqrt(np.mean([r["error_raw"] ** 2 for r in recent]))
            print(f"  [{test_date.date()}] {i+1}/{len(test_dates)} | "
                  f"pred={y_hat:.2f}% actual={y_te:.2f}% "
                  f"rolling-RMSE={recent_rmse:.3f}%")
 
    return (pd.DataFrame(results).set_index("date"),
            pd.DataFrame(shap_records).set_index("date"),
            pd.DataFrame(tuning_log))
 
 

## 6. SHAP Aggregation into Economic Blocks

`aggregate_to_blocks` maps feature-level SHAP values to the 7 economic blocks  
defined in `FEATURE_GROUPS`, producing two parallel time series per block:

- **Signed** (`group_signed`): $C_{kt} = \sum_{j \in B_k} \phi_{j,t}$  
  Preserves direction — positive = upward pressure on inflation, negative = downward.  
  Used to assess whether a block is pushing the forecast above or below the baseline(Myrland, 2026).

- **Absolute** (`group_abs`): $|C_{kt}| = \sum_{j \in B_k} |\phi_{j,t}|$  
  Captures predictive *importance* regardless of sign.  
  Used to compute attribution shares $A_{kt}$ that are directly comparable to  
  the Walker absolute centred contributions in the thesis (Myrland, 2026).

Both global (full-period) and per-regime summaries are computed and exported.

In [ ]:
def aggregate_to_blocks(shap_df: pd.DataFrame, feature_cols: List[str]):
    """Sum SHAP values within each economic block; emit signed and abs tables
    plus global and per-regime summaries. Output schema kept identical to the
    original notebook so downstream R reads them unchanged."""
    sf = shap_df[feature_cols].copy()
 
    group_signed = pd.DataFrame(index=shap_df.index)
    group_abs    = pd.DataFrame(index=shap_df.index)
    for g, members in FEATURE_GROUPS.items():
        present = [c for c in members if c in sf.columns]
        if present:
            group_signed[g] = sf[present].sum(axis=1)
            group_abs[g]    = sf[present].abs().sum(axis=1)
 
    meta_signed = ["actual", "predicted_raw", "shap_base",
                   "pred_reconstructed", "reconstruction_error", "regime"]
    for c in meta_signed:
        group_signed.insert(0, c, shap_df[c])
    group_signed = group_signed[meta_signed + BLOCK_NAMES]
    group_signed["signed_sum_excl_base"] = group_signed[BLOCK_NAMES].sum(axis=1)
    group_signed["pred_from_groups"] = (
        group_signed["shap_base"] + group_signed["signed_sum_excl_base"]
    )
 
    meta_abs = ["actual", "predicted_raw", "regime"]
    for c in meta_abs:
        group_abs.insert(0, c, shap_df[c])
    group_abs = group_abs[meta_abs + BLOCK_NAMES]
    group_abs["abs_sum_total"] = group_abs[BLOCK_NAMES].sum(axis=1)
 
    # Global
    global_signed = group_signed[BLOCK_NAMES].mean().rename("mean_signed")
    global_abs    = group_abs[BLOCK_NAMES].mean().rename("mean_abs")
    global_summary = pd.concat([global_signed, global_abs], axis=1)
    global_summary["abs_share_pct"] = (
        100 * global_summary["mean_abs"] / global_summary["mean_abs"].sum()
    )
    global_summary = global_summary.sort_values("mean_abs", ascending=False)
 
    # Regime
    rows = []
    for regime, sub in group_signed.groupby("regime"):
        sub_abs = group_abs[group_abs["regime"] == regime]
        for g in BLOCK_NAMES:
            rows.append({
                "regime": regime, "group": g,
                "mean_signed": float(sub[g].mean()),
                "mean_abs":    float(sub_abs[g].mean()),
            })
    regime_summary = pd.DataFrame(rows)
    if not regime_summary.empty:
        regime_summary["abs_share_pct"] = (
            regime_summary.groupby("regime")["mean_abs"]
            .transform(lambda s: 100 * s / s.sum())
        )
 
    return group_signed, group_abs, global_summary, regime_summary

## 7. Evaluation Metrics and File Export

### Metrics
Point forecast accuracy is measured against two benchmarks:
- **XGBoost vs. realised inflation** — RMSE, MAE, R², bias  
- **Random walk vs. realised inflation** — RMSE, bias  

Bias (`predicted − actual`) is reported separately from RMSE because systematic  
under- or overprediction is economically informative, particularly during the  
energy-crisis period when tree models cannot extrapolate beyond the training range.

### Output files written to `results_harmonized/`

| File | Contents |
|------|----------|
| `predictions.csv` | Actuals, raw predictions, RW benchmark, regime labels |
| `shap_signed.csv` | Feature-level signed SHAP values per origin |
| `shap_group_signed.csv` | Block-level signed contributions |
| `shap_group_abs.csv` | Block-level absolute contributions |
| `shap_group_global_summary.csv` | Full-period mean signed and absolute shares |
| `shap_group_regime_summary.csv` | Per-regime averages |
| `tuning_log.csv` | Optuna best params + half-life per re-tune event |
| `results.json` | Summary metrics for both specs |

These files are read directly by the R comparison script.

In [ ]:
def evaluation_metrics(spec: str, results_df: pd.DataFrame,
                       shap_df: pd.DataFrame, feature_cols: List[str]) -> Dict:
    raw_rmse = float(np.sqrt(mean_squared_error(results_df["actual"],
                                                results_df["predicted_raw"])))
    raw_mae  = float(mean_absolute_error(results_df["actual"],
                                         results_df["predicted_raw"]))
    raw_r2   = float(r2_score(results_df["actual"], results_df["predicted_raw"]))
    raw_bias = float((results_df["predicted_raw"] - results_df["actual"]).mean())
 
    rw_rmse  = float(np.sqrt(mean_squared_error(results_df["actual"],
                                                results_df["y_rw"])))
    rw_bias  = float((results_df["y_rw"] - results_df["actual"]).mean())
    return {
        "spec": spec,
        "horizon": HORIZON,
        "target": f"YoY inflation rate (%, {HORIZON}-month ahead)",
        "method": "Harmonised XGBoost with Walker-matched lag1 inputs and raw SHAP",
        "test_start": TEST_START,
        "retune_every": RETUNE_EVERY,
        "optuna_trials": OPTUNA_TRIALS,
        "min_train_n": MIN_TRAIN_N,
        "feature_set": feature_cols,
        "n_features": len(feature_cols),
        "n_test": int(len(results_df)),
        "raw_test": {"rmse": raw_rmse, "mae": raw_mae, "r2": raw_r2,
                     "bias_model_minus_actual": raw_bias},
        "random_walk": {"rmse": rw_rmse, "bias_rw_minus_actual": rw_bias},
        "reconstruction": {
            "max_abs_error":  float(np.abs(shap_df["reconstruction_error"]).max()),
            "mean_abs_error": float(np.abs(shap_df["reconstruction_error"]).mean()),
        },
    }
 
 
def save_outputs(spec: str, prefix: str,
                 results_df, shap_df, group_signed, group_abs,
                 global_summary, regime_summary, tuning_df,
                 feature_cols, metrics):
    pairs = [
        ("predictions.csv",                results_df,       True),
        ("shap_signed.csv",                shap_df,          True),
        ("shap_group_signed.csv",          group_signed,     True),
        ("shap_group_abs.csv",             group_abs,        True),
        ("shap_group_global_summary.csv",  global_summary,   True),
        ("shap_group_regime_summary.csv",  regime_summary,   False),
        ("tuning_log.csv",                 tuning_df,        False),
    ]
    for suffix, obj, has_index in pairs:
        path = RESULTS_DIR / f"{prefix}{suffix}"
        obj.to_csv(path) if has_index else obj.to_csv(path, index=False)
 
    feature_map = pd.DataFrame({
        "feature": feature_cols,
        "group":   [feature_group(c) for c in feature_cols],
    })
    feature_map.to_csv(RESULTS_DIR / f"{prefix}feature_map.csv", index=False)
 
    with open(RESULTS_DIR / f"{prefix}results.json", "w") as f:
        json.dump(metrics, f, indent=4, default=str)
 
    print(f"[{spec}] saved {len(pairs) + 2} files with prefix '{prefix}'")
 
 

## 8. Driver — Run Both Specifications

`run_spec` orchestrates the full pipeline for a single specification:  
data validation → feature engineering → backtest → block aggregation → export.

`main` runs both `main` (NOK) and `usd_rob` (USD) specifications sequentially  
and prints a summary comparison table.

In [ ]:
def run_spec(spec: str, df_raw: pd.DataFrame) -> Dict:
    oil_col, import_col, eksport_col, prefix = SPECS[spec]
    print("\n" + "=" * 70)
    print(f"SPEC: {spec}  (oil={oil_col}, import={import_col}, "
          f"eksport={eksport_col})")
    print("=" * 70)
 
    needed = ["kpi", "ppi", "unemployment", oil_col, "usd_nok", "eur_nok",
              "rente", import_col, eksport_col]
    missing = [c for c in needed if c not in df_raw.columns]
    if missing:
        raise ValueError(f"master_data.csv is missing columns: {missing}")
 
    df = df_raw[needed].apply(pd.to_numeric, errors="coerce")
 
    X_full, y_full, _ = build_features(
        df, oil_col=oil_col, import_col=import_col,
        eksport_col=eksport_col, h=HORIZON,
    )
    feature_cols = list(X_full.columns)
 
    # Joint completeness on features and target
    common_idx = X_full.index.intersection(y_full.dropna().index)
    X_full, y_full = X_full.loc[common_idx], y_full.loc[common_idx]
    mask = ~(X_full.isnull().any(axis=1) | y_full.isnull())
    X_full, y_full = X_full.loc[mask], y_full.loc[mask]
 
    print(f"Observations after cleaning: {len(X_full)}  "
          f"({X_full.index.min().date()} – {X_full.index.max().date()})")
 
    results_df, shap_df, tuning_df = run_backtest(
        X_full, y_full, feature_cols=feature_cols,
        test_start=TEST_START, retune_every=RETUNE_EVERY,
        optuna_trials=OPTUNA_TRIALS, min_train=MIN_TRAIN_N,
        random_state=RANDOM_STATE,
    )
 
    group_signed, group_abs, global_summary, regime_summary = \
        aggregate_to_blocks(shap_df, feature_cols)
 
    metrics = evaluation_metrics(spec, results_df, shap_df, feature_cols)
    save_outputs(spec, prefix, results_df, shap_df, group_signed, group_abs,
                 global_summary, regime_summary, tuning_df, feature_cols,
                 metrics)
    return metrics
 
 
def main(specs_to_run=("main", "usd_rob")):
    df_raw = pd.read_csv(DATA_FILE)
    df_raw["Date"] = pd.to_datetime(df_raw["Date"])
    df_raw = df_raw.set_index("Date").sort_index()
    df_raw = df_raw.replace("NA", np.nan)
    print(f"Loaded {DATA_FILE}: {len(df_raw)} rows, "
          f"{df_raw.index.min().date()} – {df_raw.index.max().date()}")
 
    summaries = {spec: run_spec(spec, df_raw) for spec in specs_to_run}
 
    print("\n" + "=" * 70)
    for spec, m in summaries.items():
        print(f"{spec:8s}  RMSE={m['raw_test']['rmse']:.4f}  "
              f"bias={m['raw_test']['bias_model_minus_actual']:+.4f}  "
              f"recon_max={m['reconstruction']['max_abs_error']:.2e}")
 
 
if __name__ == "__main__":
    main()

Loaded master_data.csv: 1270 rows, 1920-03-01 – 2025-12-01

SPEC: main  (oil=oil_price_nok, import=import, eksport=eksport)
Observations after cleaning: 296  (2001-02-01 – 2025-09-01)
Test period: 2020-01-01 – 2025-09-01 (69 origins)
[2020-01-01] Re-tuning (100 trials, train=227)... CV RMSE=1.0485, HL=59.0m
[2020-04-01] Re-tuning (100 trials, train=230)... CV RMSE=1.0593, HL=59.9m
[2020-07-01] Re-tuning (100 trials, train=233)... CV RMSE=1.0251, HL=43.6m
[2020-10-01] Re-tuning (100 trials, train=236)... CV RMSE=1.0097, HL=59.4m
  [2020-12-01] 12/69 | pred=1.71% actual=3.06% rolling-RMSE=0.847%
[2021-01-01] Re-tuning (100 trials, train=239)... CV RMSE=0.9833, HL=31.5m
[2021-04-01] Re-tuning (100 trials, train=242)... CV RMSE=0.9809, HL=42.8m
[2021-07-01] Re-tuning (100 trials, train=245)... CV RMSE=0.9997, HL=43.5m
[2021-10-01] Re-tuning (100 trials, train=248)... CV RMSE=1.0417, HL=50.5m
  [2021-12-01] 24/69 | pred=3.61% actual=4.54% rolling-RMSE=0.937%
[2022-01-01] Re-tuning (100 tria

## 9. Post-hoc Analysis

The cells below are run **after** the main estimation loop has completed.  
They load the saved CSV outputs and compute additional diagnostics:
- **MAPE** — mean absolute percentage error (requires non-zero actuals)
- **Attribution share tables** — signed ($S_{kt}$) and absolute ($A_{kt}$) shares  
  per block per date, computed as fraction of total absolute attribution
- **Regime summaries** — mean shares per block within each inflation episode
- **Block-level exports** — `block_signed.csv`, `block_abs.csv`, `shares.csv`,  
  `regime_summary.csv`, `global_summary.csv` for the R comparison script

These share definitions follow equations (15) and (16) in the thesis.

In [ ]:

def safe_mape(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    ok = np.isfinite(actual) & np.isfinite(predicted) & (actual != 0)
    if not ok.any():
        return np.nan
    return np.mean(np.abs((actual[ok] - predicted[ok]) / actual[ok])) * 100

rows = []
for spec, prefix in [("main", "xgb_v8_harmonized_"),
                     ("usd_rob", "xgb_harmonized_robustness_")]:
    path = RESULTS_DIR / f"{prefix}predictions.csv"
    df = pd.read_csv(path)
    err = df["predicted_raw"] - df["actual"]
    rows.append({
        "spec": spec,
        "MAE":  np.mean(np.abs(err)),
        "MAPE": safe_mape(df["actual"], df["predicted_raw"]),
        "N":    len(df),
    })

print(pd.DataFrame(rows).to_string(index=False))

   spec      MAE      MAPE  N
   main 0.900165 26.008161 69
usd_rob 0.884030 24.356795 69


In [ ]:
BLOCKS = ["AR (inflation)", "PPI / cost-push", "Monetary policy",
          "FX", "Oil", "Trade", "Labour market"]

for spec, prefix in [("main", "xgb_v8_harmonized_"),
                     ("usd_rob", "xgb_harmonized_robustness_")]:
    gs = pd.read_csv(RESULTS_DIR / f"{prefix}shap_group_signed.csv",
                     index_col="date", parse_dates=["date"])

    blocks = gs[BLOCKS]
    total_abs = blocks.abs().sum(axis=1)

    shares = pd.DataFrame(index=gs.index)
    shares["regime"] = gs["regime"]
    shares["total_abs"] = total_abs
    for b in BLOCKS:
        shares[f"S_{b}"] = np.where(total_abs > 0, blocks[b]      / total_abs, np.nan)
        shares[f"A_{b}"] = np.where(total_abs > 0, blocks[b].abs() / total_abs, np.nan)

    shares.to_csv(RESULTS_DIR / f"{prefix}shares.csv")
    print(f"[{spec}] {len(shares)} rader, gjennomsnittlige absolute shares per regime:")
    print(shares.groupby("regime")[[f"A_{b}" for b in BLOCKS]].mean().round(3))
    print()

[main] 69 rader, gjennomsnittlige absolute shares per regime:
                   A_AR (inflation)  A_PPI / cost-push  A_Monetary policy  \
regime                                                                      
COVID (2020–2021)             0.346              0.072              0.144   
Disinflation                  0.530              0.105              0.044   
Energy Crisis                 0.324              0.243              0.079   
Normalization                 0.239              0.178              0.096   

                    A_FX  A_Oil  A_Trade  A_Labour market  
regime                                                     
COVID (2020–2021)  0.204  0.070    0.117            0.046  
Disinflation       0.220  0.027    0.054            0.021  
Energy Crisis      0.060  0.080    0.165            0.050  
Normalization      0.205  0.083    0.104            0.096  

[usd_rob] 69 rader, gjennomsnittlige absolute shares per regime:
                   A_AR (inflation)  A_PPI / cost

In [ ]:
for spec, prefix in [("main", "xgb_v8_harmonized_"),
                     ("usd_rob", "xgb_harmonized_robustness_")]:

    gs = pd.read_csv(RESULTS_DIR / f"{prefix}shap_group_signed.csv",
                     index_col="date", parse_dates=["date"])

    # --- block_signed: én rad per dato, signed C_kt + baseline + regime
    block_signed = gs[BLOCKS + ["shap_base", "regime"]].rename(
        columns={"shap_base": "baseline_SHAP"}
    )
    block_signed.to_csv(RESULTS_DIR / f"{prefix}block_signed.csv")

    # --- block_abs: |C_kt|
    block_abs = gs[BLOCKS].abs().assign(regime=gs["regime"])
    block_abs.to_csv(RESULTS_DIR / f"{prefix}block_abs.csv")

    # --- shares: S_kt og A_kt rad-for-rad
    blocks = gs[BLOCKS]
    total_abs = blocks.abs().sum(axis=1)

    shares = block_signed.copy()
    shares["total_abs"] = total_abs
    for b in BLOCKS:
        shares[f"S_{b}"] = np.where(total_abs > 0, blocks[b]      / total_abs, np.nan)
        shares[f"A_{b}"] = np.where(total_abs > 0, blocks[b].abs() / total_abs, np.nan)
    shares.to_csv(RESULTS_DIR / f"{prefix}shares.csv")

    # --- regime_summary: snitt av S_ og A_ shares per regime
    share_cols = [c for c in shares.columns if c.startswith(("S_", "A_"))]
    regime_summary = (shares.groupby("regime")[share_cols]
                            .mean()
                            .assign(n_obs=shares.groupby("regime").size())
                            .reset_index())
    regime_summary.to_csv(RESULTS_DIR / f"{prefix}regime_summary.csv", index=False)

    # --- global_summary: snitt over hele perioden
    global_summary = shares[share_cols].mean().to_frame().T
    global_summary.to_csv(RESULTS_DIR / f"{prefix}global_summary.csv", index=False)

    # Konsoll-oversikt
    print(f"\n=== {spec} ===")
    print("Mean signed shares (hele perioden):")
    print(global_summary[[f"S_{b}" for b in BLOCKS]].round(3).T)
    print("\nMean absolute shares per regime:")
    print(regime_summary.set_index("regime")[[f"A_{b}" for b in BLOCKS]].round(3))


=== main ===
Mean signed shares (hele perioden):
                       0
S_AR (inflation)   0.144
S_PPI / cost-push  0.015
S_Monetary policy  0.018
S_FX               0.055
S_Oil              0.005
S_Trade            0.031
S_Labour market   -0.006

Mean absolute shares per regime:
                   A_AR (inflation)  A_PPI / cost-push  A_Monetary policy  \
regime                                                                      
COVID (2020–2021)             0.346              0.072              0.144   
Disinflation                  0.530              0.105              0.044   
Energy Crisis                 0.324              0.243              0.079   
Normalization                 0.239              0.178              0.096   

                    A_FX  A_Oil  A_Trade  A_Labour market  
regime                                                     
COVID (2020–2021)  0.204  0.070    0.117            0.046  
Disinflation       0.220  0.027    0.054            0.021  
Energy Crisis

## References

Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A
next-generation hyperparameter optimization framework. *Proceedings of the
25th ACM SIGKDD International Conference on Knowledge Discovery & Data
Mining*, 2623–2631. https://doi.org/10.1145/3292500.3330701

Lundberg, S., & Lee, S.-I. (2017). *A unified approach to interpreting model
predictions*. arXiv. https://doi.org/10.48550/ARXIV.1705.07874

Myrland, Ø. (2026). *Harmonising feature attribution across XGBoost-SHAP and
Walker TVP models* (Research Note). SSRN.
https://doi.org/10.2139/ssrn.6639699